In [ ]:
# PyTorch functions/methods helpers

# 7.4.4
assert torch.allclose(Y_conv, Y_linear, atol=1e-6) # atol = Consider these two tensors equal if their values differ by < 1e-6

* A real CNN layer does two things at once: looks over a local spatial window, and mixes the channel values available in that window.
* Output channels are learned feature maps, each produced by its own kernel across all input channels.

# How to use this notebook

* Run the notebook from top to bottom.

* Every code block is designed to be cloud-runnable and self-contained inside this notebook.

* The drills are intentionally small: predict the shape or behavior first, run the cell, then read the assertion as the contract you must understand.

# You are done when you can

- explain channels as feature dimensions attached to spatial locations
- manually combine multiple input channels
- explain the `Conv2d` weight shape
- count parameters with and without bias
- show that a 1 by 1 convolution is a per-location channel mixer

In [1]:
import torch
from torch import nn

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)

def corr2d(X, K): # Hand-writen version of nn.functional.conv2d()
    h, w = K.shape
    out_h = X.shape[0] - h + 1 # Output height = input height − kernel height + 1
    out_w = X.shape[1] - w + 1 # Output width = input width − kernel width + 1
    Y = torch.zeros(out_h, out_w, dtype=X.dtype) # Y has shape (out_h, out_w)
    for i in range(out_h):
        for j in range(out_w):
            window = X[i:i+h, j:j+w] # Extract an h × w window (shapes of K) from X
            Y[i, j] = (window * K).sum() # Calculate the value that belongs at position (i, j) in Y with the sum of element-wise products of window and K
    return Y

def corr2d_multi_in(X, K): # Hand-writen version of unsqueeze() in nn.functional.conv2d()
    pieces = [corr2d(x, k) for x, k in zip(X, K)] # Apply corr2d independently to each input channel/kernel pair
    return torch.stack(pieces).sum(dim=0) # Stack the channel-wise results, then sum across channels to produce one output feature map

def conv2d_param_count(in_channels, out_channels, kernel_h, kernel_w, bias=True): # Hand-writen version of Conv2d.parameters()
    weights = out_channels * in_channels * kernel_h * kernel_w # Calculates the number of weight parameters
    return weights + (out_channels if bias else 0) # If bias is enabled, add one bias per output channel. Otherwise, add zero

# 7.4.0 The Problem This Notebook Solves

Earlier examples used one input channel so the sliding-window idea stayed simple.

Real images and hidden CNN layers usually have multiple channels.

The channel dimension answers:

```text
what kinds of information exist at each spatial location?
```

* For RGB images, the channels are color measurements.
* For hidden layers, the channels are learned feature maps.
* A useful detector often needs to combine those measurements.
* For example, a detector might care about a red-green contrast, or a hidden-layer detector might combine edge evidence and texture evidence.

A convolutional kernel for one output channel therefore has:

```text
one spatial kernel per input channel
```

* Each channel-specific kernel creates a response map.
* Those response maps are added to form one output feature map.
* Multiple output channels repeat this process with different learned kernels.

# 7.4.1 Multiple Input Channels Are Summed Into One Response Map

For one output channel, the layer has one small spatial kernel per input channel.

Each channel produces a response map, and those maps are added.

The theory-level meaning is that one feature detector may depend on several kinds of evidence.

It is not forced to inspect red, green, blue, or hidden feature channels independently. It can learn how to combine them.

The manual function below makes that explicit:

```text
for each input channel:
    run 2D correlation
sum the channel responses
```

In [2]:
X = torch.arange(18, dtype=torch.float32).reshape(2, 3, 3)
K = torch.ones(2, 2, 2)

Y = corr2d_multi_in(X, K) # Shape result from channel 0: 2x2; shape result from channel 1: 2x2, sum them toegher: shape of 2x2

print("input shape:", shape(X))
print("kernel shape:", shape(K))
print(Y)

assert shape(Y) == (2, 2)

input shape: (2, 3, 3)
kernel shape: (2, 2, 2)
tensor([[52., 60.],
        [76., 84.]])


# 7.4.2 Verify Multi-Input-Channel Behavior With `Conv2d`

PyTorch stores convolution weights in this order:

```text
output channels, input channels, kernel height, kernel width
```

That order says exactly how to read the layer:

- choose an output feature map
- for that output map, look across all input channels
- for each input channel, use a small spatial kernel

The cell copies the manual kernel into `Conv2d` and verifies the result. This is the bridge from scratch mechanics to the library API.

In [4]:
conv = nn.Conv2d(2, 1, kernel_size=2, bias=False) # Looking for 2 input channels, 1 output channels, at the kernel_size of 2x2

with torch.no_grad():
    conv.weight[:] = K.reshape(1, 2, 2, 2) # Weight shape is 1 output channel, 2 input channels, at the kernel_size of 2x2
                                           # "I have one output feature map, and to produce it I look at 2 input channels using a 2×2 kernel on each channel"

Y_torch = conv(X.reshape(1, 2, 3, 3)) # Shape of (1, 1, 2, 2)

print("conv weight shape:", shape(conv.weight))
print(Y_torch[0, 0])
print(Y_torch.shape)

assert shape(conv.weight) == (1, 2, 2, 2)
assert torch.allclose(Y_torch[0, 0], Y)

conv weight shape: (1, 2, 2, 2)
tensor([[52., 60.],
        [76., 84.]], grad_fn=<SelectBackward0>)
torch.Size([1, 1, 2, 2])


# 7.4.3 Multiple Output Channels Mean Multiple Learned Kernels

One output channel gives one learned feature map.

Real layers need many feature maps because images contain many useful patterns.
* The output channel count is therefore the layer's width.
* More output channels mean more learned detectors and more capacity.
* They also mean more computation and memory.

The parameter count comes from:

```text
output channels * input channels * kernel height * kernel width
plus one bias per output channel if bias=True
```

The output shape separates batch, output channels, and spatial dimensions:

```text
batch, output_channels, output_height, output_width
```

In [5]:
conv = nn.Conv2d(in_channels=3, out_channels=5, kernel_size=3, padding=1, bias=True)

X = torch.zeros(4, 3, 8, 8)
Y = conv(X)

param_count = sum(p.numel() for p in conv.parameters())
expected_params = conv2d_param_count(3, 5, 3, 3, bias=True) # Weight shape is (5, 3, 3, 3), then plus 1 bias per output channel (5), so 5*3*3*3+5 = 140

print("output shape:", shape(Y)) # Shape of (4, 5, 8, 8) with batch_size = 4 from X, out_channels = 5 from Conv2d, and 8*8 from X's heights and widths maintained due to padding
print("weight shape:", shape(conv.weight))
print("parameter count:", param_count)

assert shape(Y) == (4, 5, 8, 8)
assert param_count == expected_params

output shape: (4, 5, 8, 8)
weight shape: (5, 3, 3, 3)
parameter count: 140


# 7.4.4 A 1 by 1 Convolution Mixes Channels at Each Location

A 1 by 1 convolution sounds spatially tiny, but it can be powerful because it mixes channels.

At each row and column, the layer sees the channel vector at that exact location.

It applies the same linear transformation to that vector everywhere.

Plain-English meaning:

```text
do not look at neighboring locations
recombine the feature types at each location
optionally change the channel count
```

* This is why 1 by 1 convolutions appear in many modern CNN blocks.
* They can compress channels, expand channels, or mix feature evidence before or after spatial convolutions.

The cell proves the equivalence by rebuilding a 1 by 1 convolution as a regular linear layer applied to every pixel's channel vector.

In [6]:
torch.manual_seed(0)

conv = nn.Conv2d(3, 2, kernel_size=1, bias=False) # Takes in_channels = 3, out_channels = 2, and kernel_size = 1*1 (1x1 convolution)
linear = nn.Linear(3, 2, bias=False)

with torch.no_grad():
    linear.weight[:] = conv.weight[:, :, 0, 0] # Copy the 2×3 weights from the 1×1 Conv2d kernel into the Linear layer

X = torch.randn(1, 3, 4, 5)
Y_conv = conv(X)

# This is the recreation of Y_conv with 1x1 convolution (kernel_size) but linearly (Y_linear)
# X shape of (1, 3, 4, 5) -> permute(0, 2, 3, 1)
# X's new shape of (1, 4, 5, 3) -> reshape(-1, 3)
# X's new shape of (1*4*5, 3) -> (20, 3)
# X's final shape of (20, 3) @ linear.weight.T of (3, 2)
# Y's shape of (20, 2) -> reshape (1, 4, 5, 2)
# Y's new shape of (1, 4, 5, 2) -> permute(0, 3, 1, 2)
# Y's final shape of (1, 2, 4, 5)
pixels = X.permute(0, 2, 3, 1).reshape(-1, 3)
Y_linear = linear(pixels).reshape(1, 4, 5, 2).permute(0, 3, 1, 2)

print("conv output shape:", shape(Y_conv)) # Shape of (1, 2, 4, 5) with batch_size = 1 from X, out_channels = 2 from Conv2d, and 4*5 from X's heights and widths
print("linear-rebuilt output shape:", shape(Y_linear))
print("max difference:", float((Y_conv - Y_linear).abs().max()))

assert torch.allclose(Y_conv, Y_linear, atol=1e-6) # atol = Consider these two tensors equal if their values differ by < 1e-6

conv output shape: (1, 2, 4, 5)
linear-rebuilt output shape: (1, 2, 4, 5)
max difference: 0.0


/tmp/ipykernel_878/2981427347.py:17: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("max difference:", float((Y_conv - Y_linear).abs().max()))


# 7.4.5 Break It Deliberately: Use NHWC Instead of NCHW

PyTorch `Conv2d` expects:

```text
NCHW: batch, channels, height, width
```

Some other libraries or image utilities use:

```text
NHWC: batch, height, width, channels
```

* These shapes can contain the same raw numbers but mean different things.
* If you pass NHWC to PyTorch without rearranging dimensions, PyTorch interprets the height as the channel count.

The theory-level mistake is not respecting the semantic meaning of tensor axes.

Shape numbers alone are not enough; axis meaning matters.

## NCHW vs NHWC

`NCHW` = `(batch, channels, height, width)` — PyTorch's usual `Conv2d` format.

`NHWC` = `(batch, height, width, channels)` — TensorFlow, JAX, Keras all use this image layout.

They contain the same dimensions; only their ordering differs. `permute()` can rearrange between them.

In [7]:
conv = nn.Conv2d(3, 4, kernel_size=3)
bad_X = torch.zeros(1, 8, 8, 3)

try:
    conv(bad_X) # Would have been fine with bad_X's shape as (1, 3, 8, 8)
except RuntimeError as err:
    print(type(err).__name__)
    print(str(err).splitlines()[0])
else:
    raise AssertionError("NHWC input should fail for this Conv2d layer.")

RuntimeError
Given groups=1, weight of size [4, 3, 3, 3], expected input[1, 8, 8, 3] to have 3 channels, but got 8 channels instead


# 7.4 Checkpoint

Answer these before moving on.

Short markdown answers in the notebook are enough; the chapter does not need a separate notes file.

1. Why does a useful detector often need to combine several input channels?
> Because a useful feature can depend on a combination of multiple input channels

2. Why does a convolution kernel span all input channels?
> Because each output feature can depend on information from every input channel, so the kernel has weights for every input channel

3. What does each output channel represent?
> Each output channel represents a learned feature map produced by a different filter, potentially detecting a different pattern

4. How do output channels relate to model capacity and computation?
> More output channels allow the model to learn more different features, increasing model capacity but also increasing computation and parameters

5. Why does a 1 by 1 convolution still have many parameters when channel counts are large?
> A 1×1 kernel has few spatial weights, but it still has a separate weight for every input-channel/output-channel combination, so large channel counts can produce many parameters

6. What is wrong with passing NHWC tensors to PyTorch `Conv2d`?
> PyTorch uses NCHW so formatting like NHWC from TensorFlow/Keras/JAX would not work